# Algorithm Comparison Using GridSearchCV

### Section 1 Imports

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler, OrdinalEncoder
from sklearn.metrics import f1_score, r2_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

### Section 2 Load splits

In [2]:
def load_splits(data_dir='../data/'):
    train = pd.read_parquet(os.path.join(data_dir, 'train.parquet'))
    val = pd.read_parquet(os.path.join(data_dir, 'val.parquet'))
    test = pd.read_parquet(os.path.join(data_dir, 'test.parquet'))
    return train, val, test

train_df, val_df, test_df = load_splits()
print(f"Splits loaded. Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Splits loaded. Train: (922424, 23), Val: (197663, 23), Test: (197663, 23)


### Section 3 Create target columns

In [3]:
# Model A Target: has_wait
for df in [train_df, val_df, test_df]:
    df['has_wait'] = (df['estimated_wait_time_mins'] > 0).astype(int)

# Model B Target: high_utilization
threshold = 0.5 
for df in [train_df, val_df, test_df]:
    df['high_utilization'] = (df['utilization_rate'] >= threshold).astype(int)

print("Targets created.")

Targets created.


### Section 4 Define features

In [ ]:
# Model A
features_a = [
    'power_output_kw', 'ports_total', 'traffic_congestion_index', 
    'is_peak_hour', 'charger_type', 'pricing_type'
]
# Model B
features_b = [
    'hour_of_day', 'day_of_week', 'traffic_congestion_index', 'is_peak_hour'
]
# Model C
cat_features_c = ['charger_type']
num_features_c = ['power_output_kw']
features_c = num_features_c + cat_features_c

# Model D
cat_features_d = ['charger_type', 'pricing_type', 'network']
num_features_d = ['hour_of_day', 'is_peak_hour', 'power_output_kw', 'ports_total'] # Removed utilization_rate
features_d = num_features_d + cat_features_d

### Section 5 Define preprocessors

In [5]:
# Preprocessor A & B (Simple Pipeline)
def get_preprocessor_ab():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])

# Preprocessor C
preprocessor_c = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())]), num_features_c),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_features_c)
])

# Preprocessor D
preprocessor_d = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features_d),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_features_d)
])

### Helper Functions for Evaluation

In [ ]:
def run_grid_search(name, pipeline, param_grid, X_train, y_train, scoring):
    print(f"Running GridSearchCV for {name}...")
    grid = GridSearchCV(
        estimator  = pipeline,
        param_grid = param_grid,
        cv         = 3,
        scoring    = scoring,
        n_jobs     = -1,
        verbose    = 1,
        refit      = True
    )
    grid.fit(X_train, y_train)
    return grid

def print_comparison_table(model_name, target, metric_name, results, X_tr, y_tr, X_va, y_va, X_te, y_te, metric_fn, apply_rule=None):

    print(f"{'Algorithm':<15} {'Train':<10} {'Val':<10} {'Test':<10} {'Gap':<10}")
    
    scores = {}
    for alg in ['XGBoost', 'LightGBM']:
        best_model = results[alg].best_estimator_
        tr_preds = best_model.predict(X_tr)
        va_preds = best_model.predict(X_va)
        te_preds = best_model.predict(X_te)
        
        if apply_rule:
            tr_preds = apply_rule(train_df, tr_preds)
            va_preds = apply_rule(val_df, va_preds)
            te_preds = apply_rule(test_df, te_preds)
            
        tr_s = metric_fn(y_tr, tr_preds)
        va_s = metric_fn(y_va, va_preds)
        te_s = metric_fn(y_te, te_preds)
        gap = abs(tr_s - va_s)
        
        print(f"{alg:<15} {tr_s:<10.4f} {va_s:<10.4f} {te_s:<10.4f} {gap:<10.4f}")
        scores[alg] = va_s
    
    return scores

### Section 6 GridSearchCV comparison for Model A

In [7]:
X_train_a, y_train_a = train_df[features_a], train_df['has_wait']
X_val_a, y_val_a = val_df[features_a], val_df['has_wait']
X_test_a, y_test_a = test_df[features_a], test_df['has_wait']

results_a = {}
params_xgb = {'classifier__n_estimators': [100, 300], 'classifier__max_depth': [4, 6]}
params_lgbm = {'classifier__n_estimators': [100, 300], 'classifier__max_depth': [4, 6]}

results_a['XGBoost'] = run_grid_search('XGBoost', Pipeline([('prep', get_preprocessor_ab()), ('classifier', XGBClassifier(random_state=42))]), params_xgb, X_train_a, y_train_a, 'f1_weighted')
results_a['LightGBM'] = run_grid_search('LightGBM', Pipeline([('prep', get_preprocessor_ab()), ('classifier', LGBMClassifier(random_state=42))]), params_lgbm, X_train_a, y_train_a, 'f1_weighted')

val_scores_a = print_comparison_table('Model A', 'has_wait', 'F1 Weighted', results_a, X_train_a, y_train_a, X_val_a, y_val_a, X_test_a, y_test_a, lambda y,p: f1_score(y,p,average='weighted'))

print("\nBest params per algorithm:")
for alg in ['XGBoost', 'LightGBM']:
    print(f"{alg:<12} : {results_a[alg].best_params_}")

Running GridSearchCV for XGBoost...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Running GridSearchCV for LightGBM...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[LightGBM] [Info] Number of positive: 59122, number of negative: 863302
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008304 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 290
[LightGBM] [Info] Number of data points in the train set: 922424, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.064094 -> initscore=-2.681161
[LightGBM] [Info] Start training from score -2.681161
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

### Section 7 GridSearchCV comparison for Model B

In [8]:
X_train_b, y_train_b = train_df[features_b], train_df['high_utilization']
X_val_b, y_val_b = val_df[features_b], val_df['high_utilization']
X_test_b, y_test_b = test_df[features_b], test_df['high_utilization']

results_b = {}
results_b['XGBoost'] = run_grid_search('XGBoost', Pipeline([('prep', get_preprocessor_ab()), ('classifier', XGBClassifier(random_state=42))]), params_xgb, X_train_b, y_train_b, 'f1_weighted')
results_b['LightGBM'] = run_grid_search('LightGBM', Pipeline([('prep', get_preprocessor_ab()), ('classifier', LGBMClassifier(random_state=42))]), params_lgbm, X_train_b, y_train_b, 'f1_weighted')

val_scores_b = print_comparison_table('Model B', 'high_utilization', 'F1 Weighted', results_b, X_train_b, y_train_b, X_val_b, y_val_b, X_test_b, y_test_b, lambda y,p: f1_score(y,p,average='weighted'))

print("\nBest params per algorithm:")
for alg in ['XGBoost', 'LightGBM']:
    print(f"{alg:<12} : {results_b[alg].best_params_}")

Running GridSearchCV for XGBoost...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Running GridSearchCV for LightGBM...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[LightGBM] [Info] Number of positive: 376288, number of negative: 546136
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013790 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 99
[LightGBM] [Info] Number of data points in the train set: 922424, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.407934 -> initscore=-0.372513
[LightGBM] [Info] Start training from score -0.372513

â”€â”€ Model B â€” high_utilization (F1 Weighted) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

Algorithm       Train      Val        Test       Gap       
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â

### Section 8 GridSearchCV comparison for Model C

In [9]:
X_train_c, y_train_c = train_df[features_c], train_df['avg_session_duration_mins']
X_val_c, y_val_c = val_df[features_c], val_df['avg_session_duration_mins']
X_test_c, y_test_c = test_df[features_c], test_df['avg_session_duration_mins']

results_c = {}
params_xgb_reg = {k.replace('classifier', 'regressor'): v for k, v in params_xgb.items()}
params_lgbm_reg = {k.replace('classifier', 'regressor'): v for k, v in params_lgbm.items()}

results_c['XGBoost'] = run_grid_search('XGBoost', Pipeline([('prep', preprocessor_c), ('regressor', XGBRegressor(random_state=42))]), params_xgb_reg, X_train_c, y_train_c, 'r2')
results_c['LightGBM'] = run_grid_search('LightGBM', Pipeline([('prep', preprocessor_c), ('regressor', LGBMRegressor(random_state=42))]), params_lgbm_reg, X_train_c, y_train_c, 'r2')

val_scores_c = print_comparison_table('Model C', 'duration', 'R2', results_c, X_train_c, y_train_c, X_val_c, y_val_c, X_test_c, y_test_c, r2_score)

print("\nBest params per algorithm:")
for alg in ['XGBoost', 'LightGBM']:
    print(f"{alg:<12} : {results_c[alg].best_params_}")

Running GridSearchCV for XGBoost...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Running GridSearchCV for LightGBM...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003170 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13
[LightGBM] [Info] Number of data points in the train set: 922424, number of used features: 2
[LightGBM] [Info] Start training from score 81.914084
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

### Section 9 GridSearchCV comparison for Model D

In [10]:
X_train_d, y_train_d = train_df[features_d], train_df['current_price']
X_val_d, y_val_d = val_df[features_d], val_df['current_price']
X_test_d, y_test_d = test_df[features_d], test_df['current_price']

def apply_free_rule(df, preds):
    return np.where(df['pricing_type'] == 'free', 0.0, np.clip(preds, 0, None))

results_d = {}
results_d['XGBoost'] = run_grid_search('XGBoost', Pipeline([('prep', preprocessor_d), ('regressor', XGBRegressor(random_state=42))]), params_xgb_reg, X_train_d, y_train_d, 'r2')
results_d['LightGBM'] = run_grid_search('LightGBM', Pipeline([('prep', preprocessor_d), ('regressor', LGBMRegressor(random_state=42))]), params_lgbm_reg, X_train_d, y_train_d, 'r2')

val_scores_d = print_comparison_table('Model D', 'price', 'R2', results_d, X_train_d, y_train_d, X_val_d, y_val_d, X_test_d, y_test_d, r2_score, apply_rule=apply_free_rule)

print("\nBest params per algorithm:")
for alg in ['XGBoost', 'LightGBM']:
    print(f"{alg:<12} : {results_d[alg].best_params_}")

Running GridSearchCV for XGBoost...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Running GridSearchCV for LightGBM...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022253 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 318
[LightGBM] [Info] Number of data points in the train set: 922424, number of used features: 8
[LightGBM] [Info] Start training from score 0.313571

â”€â”€ Model D â€” price (R2) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

Algorithm       Train      Val        Test       Gap       
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
XGBoost         0.9942     0.9940     0.9942     0.0001    
LightGBM        0.9930     0.9929     0.9931   

### Section 10 Final summary table

In [ ]:
summary_data = [
    ['Model A', 'has_wait', '(F1)', val_scores_a['XGBoost'], val_scores_a['LightGBM']],
    ['Model B', 'high_util', '(F1)', val_scores_b['XGBoost'], val_scores_b['LightGBM']],
    ['Model C', 'duration', '(R2)', val_scores_c['XGBoost'], val_scores_c['LightGBM']],
    ['Model D', 'price', '(R2)', val_scores_d['XGBoost'], val_scores_d['LightGBM']]
]

summary_df = pd.DataFrame(summary_data, columns=['Model', 'Target', 'Metric', 'XGBoost', 'LightGBM'])

print("\nFinal Comparison (Val Scores)\n")
print(f"{'Model':<8} {'Target':<15} {'XGBoost':<10} {'LightGBM':<10}")
for _, row in summary_df.iterrows():
    print(f"{row['Model']:<8} {row['Target']+' '+row['Metric']:<15} {row['XGBoost']:<10.4f} {row['LightGBM']:<10.4f}")

print("\nUser reviews this table and selects\none algorithm per model in next phase.")


â”€â”€ Final Comparison (Val Scores) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

Model    Target          XGBoost    LightGBM  
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
Model A  has_wait (F1)   0.9807     0.9807    
Model B  high_util (F1)  0.9840     0.9841    
Model C  duration (R2)   0.8840     0.8840    
Model D  price (R2)      0.9940     0.9929    

User reviews this table and selects
one algorithm per model in next phase.


In [ ]:
comparison_table = """
Model A has_wait (F1)
Algorithm      Train     Val       Test      Gap
XGBoost        0.9804    0.9807    0.9811    0.0003
LightGBM       0.9804    0.9807    0.9811    0.0003
RandomForest   0.9804    0.9807    0.9811   -0.0003
â”€â”€ Identical â€” pick any

Model B high_utilization (F1)
Algorithm      Train     Val       Test      Gap
XGBoost        0.9840    0.9840    0.9837    0.0000
LightGBM       0.9840    0.9841    0.9838    0.0001
RandomForest   0.9085    0.9089    0.9079   -0.0004
â”€â”€ XGBoost and LightGBM massively better than RF

Model C session duration (RÂ²)
Algorithm      Train     Val       Test      Gap
XGBoost        0.8835    0.8840    0.8836   -0.0005
LightGBM       0.8835    0.8840    0.8836   -0.0005
RandomForest   0.8835    0.8840    0.8836   -0.0005
â”€â”€ Identical across all three

Model D current price (RÂ²)
Algorithm      Train     Val       Test      Gap
XGBoost        0.9942    0.9940    0.9942    0.0001
LightGBM       0.9930    0.9929    0.9931    0.0001
RandomForest   0.9738    0.9735    0.9740    0.0003
â”€â”€ XGBoost clearly best
"""
print(comparison_table)


Model A â€” has_wait (F1)
Algorithm      Train     Val       Test      Gap
XGBoost        0.9804    0.9807    0.9811    0.0003
LightGBM       0.9804    0.9807    0.9811    0.0003
RandomForest   0.9804    0.9807    0.9811   -0.0003
â”€â”€ Identical â€” pick any

Model B â€” high_utilization (F1)
Algorithm      Train     Val       Test      Gap
XGBoost        0.9840    0.9840    0.9837    0.0000
LightGBM       0.9840    0.9841    0.9838    0.0001
RandomForest   0.9085    0.9089    0.9079   -0.0004
â”€â”€ XGBoost and LightGBM massively better than RF

Model C â€” session duration (RÂ²)
Algorithm      Train     Val       Test      Gap
XGBoost        0.8835    0.8840    0.8836   -0.0005
LightGBM       0.8835    0.8840    0.8836   -0.0005
RandomForest   0.8835    0.8840    0.8836   -0.0005
â”€â”€ Identical across all three

Model D â€” current price (RÂ²)
Algorithm      Train     Val       Test      Gap
XGBoost        0.9942    0.9940    0.9942    0.0001
LightGBM       0.9930    0.9929    0